# Static spatial layers — Iberian wildfire forecasting

Aggregates three external datasets to the project 0.25° grid (32 lat × 54 lon)
covering the Iberian Peninsula:

| Layer | Source | Output channels |
|---|---|---|
| Land cover | ESA WorldCover 2021 v200 (10 m) | 7 fuel-class fractions |
| Topography | Copernicus DEM GLO-30 (~30 m) | mean elevation, std elevation, mean slope |
| Vegetation | MODIS MOD13A3 NDVI (1 km, monthly) | static climatology + dynamic anomaly |

Outputs are NetCDF files on the same grid as the ERA5 climate cube, suitable
for direct stacking downstream.

**Conventions**
- Grid: lat ∈ {36.25, 36.50, …, 44.00}, lon ∈ {−10.00, −9.75, …, 3.25}, node-centred 0.25°.
- Fractional / area-weighted aggregation everywhere it makes sense.
- Train-window-only statistics for the NDVI climatology (configured below).
- Every section is independent and resumable — re-running the notebook only
  re-downloads files that are missing.

**Run-time / disk budget (rough)**:
- WorldCover: ~5 GB raw, ~30 min on a fast connection.
- Copernicus DEM: ~3.5 GB raw, ~15 min.
- MODIS NDVI: ~5 GB raw across ~1,200 granules, ~2 h. Requires a free [NASA
  Earthdata Login](https://urs.earthdata.nasa.gov/).


## 0. Setup

Configuration shared across all three sections. Run this once before anything else.

In [ ]:
import os
import re
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import xarray as xr
import rioxarray  # noqa: F401  (registers .rio accessor)
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import Affine
import requests
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

In [17]:
# ---- Project grid ------------
DATA_ROOT = Path("./data").resolve()
RAW_DIR   = DATA_ROOT / "raw"
PROC_DIR  = DATA_ROOT / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Bounding box (lon_min, lat_min, lon_max, lat_max).
BBOX = (-10.0, 36.25, 3.25, 44.0)

# Grid points (node-centred). Each point represents a 0.25° × 0.25° cell.
LAT = np.round(np.arange(36.25, 44.0 + 1e-9, 0.25), 4)   # 32 ascending
LON = np.round(np.arange(-10.0,  3.25 + 1e-9, 0.25), 4)  # 54 ascending
assert LAT.size == 32 and LON.size == 54, (LAT.size, LON.size)

# rasterio destination transform — origin = upper-left corner of upper-left pixel.
DST_TRANSFORM = Affine(0.25, 0.0, LON[0]  - 0.125,
                        0.0, -0.25, LAT[-1] + 0.125)
DST_SHAPE = (LAT.size, LON.size)
DST_CRS   = "EPSG:4326"

print(f"Target grid: {LAT.size} × {LON.size} cells")
print(f"  lat: {LAT[0]:.2f} → {LAT[-1]:.2f}")
print(f"  lon: {LON[0]:.2f} → {LON[-1]:.2f}")

Target grid: 32 × 54 cells
  lat: 36.25 → 44.00
  lon: -10.00 → 3.25


In [ ]:
def stream_download(url: str, dest: Path, chunk: int = 1 << 20) -> Path:
    """Resumable HTTP download with progress bar. Skips if dest already exists."""
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(tmp, "wb") as f, tqdm(
            total=total, unit="B", unit_scale=True,
            desc=dest.name, leave=False
        ) as bar:
            for buf in r.iter_content(chunk_size=chunk):
                f.write(buf)
                bar.update(len(buf))
    tmp.rename(dest)
    return dest


def reproject_to_target(
    src_data: np.ndarray,
    src_transform: Affine,
    src_crs,
    *,
    resampling: Resampling = Resampling.average,
) -> np.ndarray:
    """Reproject a 2-D source array onto the project grid.

    The trailing `[::-1]` flips the latitude axis so the returned array is
    aligned with our ascending `LAT` coordinate (rasterio writes north-up,
    but the project convention is south-first / ascending lat)."""
    out = np.zeros(DST_SHAPE, dtype=np.float32)
    reproject(
        source=np.asarray(src_data, dtype=np.float32),
        destination=out,
        src_transform=src_transform, src_crs=src_crs,
        dst_transform=DST_TRANSFORM, dst_crs=DST_CRS,
        resampling=resampling,
    )
    return out[::-1]

## 1. ESA WorldCover — fractional fuel cover

WorldCover 2021 v200 is hosted as Cloud-Optimized GeoTIFFs on AWS Open Data.
Tiles are 3° × 3°, named by SW corner. The Iberian bbox needs **18 tiles**
(3 latitude bands × 6 longitude bands).

For each fuel group we treat the categorical raster as a binary mask
(`pixel ∈ classes_in_group`) at native 10 m resolution, then use
`Resampling.average` to obtain area-weighted fractional cover on the 0.25°
grid. WorldCover tiles are non-overlapping, so per-tile contributions sum
correctly across the domain.

In [ ]:
WC_BASE = "https://esa-worldcover.s3.amazonaws.com/v200/2021/map"
WC_DIR  = RAW_DIR / "worldcover"


def wc_tile_name(lat_sw: int, lon_sw: int) -> str:
    ns = "N" if lat_sw >= 0 else "S"
    ew = "E" if lon_sw >= 0 else "W"
    return f"{ns}{abs(lat_sw):02d}{ew}{abs(lon_sw):03d}"


def wc_tiles_for_bbox(bbox):
    """3°×3° tile SW corners that intersect the bbox."""
    lon_min, lat_min, lon_max, lat_max = bbox
    lat_lo = int(np.floor(lat_min / 3.0)) * 3
    lat_hi = int(np.floor((lat_max - 1e-9) / 3.0)) * 3
    lon_lo = int(np.floor(lon_min / 3.0)) * 3
    lon_hi = int(np.floor((lon_max - 1e-9) / 3.0)) * 3
    for la in range(lat_lo, lat_hi + 1, 3):
        for lo in range(lon_lo, lon_hi + 1, 3):
            yield la, lo


WC_TILES = list(wc_tiles_for_bbox(BBOX))
print(f"Need {len(WC_TILES)} WorldCover tiles "
      f"({len({t[0] for t in WC_TILES})} lat × {len({t[1] for t in WC_TILES})} lon)")

In [ ]:
# Download (each tile ~250 MB; total ~5 GB).
wc_paths = []
for la, lo in tqdm(WC_TILES, desc="WorldCover tiles"):
    name = wc_tile_name(la, lo)
    url  = f"{WC_BASE}/ESA_WorldCover_10m_2021_v200_{name}_Map.tif"
    dst  = WC_DIR / f"ESA_WorldCover_10m_2021_v200_{name}_Map.tif"
    stream_download(url, dst)
    wc_paths.append(dst)

print(f"Have {len(wc_paths)} files in {WC_DIR}")

In [ ]:
# WorldCover class codes (from the v200 product user manual).
WC_CLASSES = {
     10: "tree_cover",       20: "shrubland",       30: "grassland",
     40: "cropland",         50: "built_up",        60: "bare_sparse",
     70: "snow_ice",         80: "water",           90: "wetland",
     95: "mangroves",       100: "moss_lichen",
}

# Fuel grouping: which raw classes feed each output channel.
# Rationale (see survey + decision_log): collapse 11 classes into 7 fuel-relevant
# groups. Mediterranean shrubland (maquis/garrigue) is fire-prone and kept
# distinct; cropland separated for stubble fires; built-up retained because it
# matters for WUI exposure and acts as a fuel break.
FUEL_GROUPS = {
    "forest":    [10],
    "shrubland": [20],
    "grassland": [30, 100],     # grassland + moss/lichen
    "cropland":  [40],
    "built_up":  [50],
    "bare":      [60, 70],      # bare ground + snow/ice
    "wet":       [80, 90, 95],  # water + wetland + mangrove (non-burnable)
}
# Sanity: every WorldCover class is mapped exactly once.
assert sorted(c for codes in FUEL_GROUPS.values() for c in codes) == sorted(WC_CLASSES)

In [ ]:
def fractional_cover(tile_path: Path, target_codes: list[int]) -> np.ndarray:
    """Reproject a binary class-membership mask to the target grid."""
    with rasterio.open(tile_path) as src:
        data = src.read(1)
        mask = np.isin(data, target_codes).astype(np.float32)
        return reproject_to_target(mask, src.transform, src.crs,
                                    resampling=Resampling.average)


# Build (n_groups, lat, lon) array.
fuel_arr = np.zeros((len(FUEL_GROUPS), *DST_SHAPE), dtype=np.float32)

for k, (group, codes) in enumerate(FUEL_GROUPS.items()):
    accum = np.zeros(DST_SHAPE, dtype=np.float32)
    for tif in tqdm(wc_paths, desc=group, leave=False):
        accum += fractional_cover(tif, codes)
    fuel_arr[k] = accum
    print(f"  {group:10s}  mean fraction = {accum.mean():.3f}, "
          f"max = {accum.max():.3f}")

# The seven fractions should sum to ≈ 1 over land. Cells over open ocean
# outside any tile sum to 0; mixed coastal cells fall in between.
total = fuel_arr.sum(axis=0)
print(f"\\nFuel-fraction sum:  min {total.min():.3f}  "
      f"max {total.max():.3f}  mean {total.mean():.3f}")

In [ ]:
# Wrap as Dataset and save.
fuel_da = xr.DataArray(
    fuel_arr,
    dims=("fuel_class", "lat", "lon"),
    coords={"fuel_class": list(FUEL_GROUPS.keys()), "lat": LAT, "lon": LON},
    name="fuel_fraction",
    attrs={
        "description": "Area-weighted fractional cover by fuel-relevant class group",
        "units": "1",
        "valid_range": [0.0, 1.0],
    },
)
fuel_ds = fuel_da.to_dataset()
fuel_ds.attrs.update(
    title="ESA WorldCover 2021 v200 — fractional fuel cover on Iberian 0.25° grid",
    grid="ERA5-aligned 0.25°, 32 × 54 nodes",
    source=WC_BASE,
    fuel_groups=", ".join(f"{k}: {v}" for k, v in FUEL_GROUPS.items()),
    bbox=str(BBOX),
)

out_wc = PROC_DIR / "static_landcover_fuelfractions.nc"
fuel_ds.to_netcdf(out_wc)
print(f"Wrote {out_wc}  ({out_wc.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Quick visual sanity check.
fig, axes = plt.subplots(2, 4, figsize=(14, 6), constrained_layout=True)
for ax, (group, _) in zip(axes.flat, FUEL_GROUPS.items()):
    fuel_da.sel(fuel_class=group).plot(
        ax=ax, vmin=0, vmax=1, cmap="viridis", add_colorbar=False
    )
    ax.set_title(group)
    ax.set_xlabel(""); ax.set_ylabel("")
axes.flat[-1].axis("off")
fig.suptitle("WorldCover fuel fractions — Iberia 0.25°")
plt.show()

## 2. Copernicus DEM GLO-30 — elevation and slope

Copernicus DEM GLO-30 (1 arc-second ≈ 30 m) from the AWS Open Data registry.
Tiles are 1° × 1°, named by SW corner. Iberia needs up to **112 tiles** (some
ocean tiles legitimately don't exist in the archive — handled gracefully).

**Processing approach.** Rather than reprojecting tile by tile and
accumulating, we combine all downloaded tiles into a single GDAL VRT mosaic
and treat it as one logical raster. Per-tile reprojection turns out to be
fragile on real GLO-30 tiles — destination cells whose centres fall on
integer-degree tile boundaries get systematically skipped — and a single-
source warp sidesteps the issue entirely.

The pipeline is:

1. `osgeo.gdal.BuildVRT` mosaics the downloaded tiles into a small XML stub.
2. `osgeo.gdal.DEMProcessing` produces a slope raster from the mosaic at
   native resolution. This uses a constant m/° scale, which biases slope
   in the longitude direction by `1/cos(lat)` — about 18 % at 40°N.
   Acceptable for a static covariate; revisit if the downstream model turns
   out to be slope-sensitive.
3. `rioxarray.reproject_match` takes the mosaic (and the slope raster, and
   `elev²` for the variance identity) onto the project grid using
   `Resampling.average`. Reads are lazy and chunked via Dask, so memory
   stays bounded even though the full mosaic is several GB at native
   resolution.

The variance identity `Var(z) = E[z²] − E[z]²` gives elevation std without
ever materialising the full mosaic.

In [ ]:
DEM_BASE = "https://copernicus-dem-30m.s3.amazonaws.com"
DEM_DIR  = RAW_DIR / "copernicus_dem"


def dem_tile_id(lat_sw: int, lon_sw: int) -> str:
    ns = "N" if lat_sw >= 0 else "S"
    ew = "E" if lon_sw >= 0 else "W"
    return f"Copernicus_DSM_COG_10_{ns}{abs(lat_sw):02d}_00_{ew}{abs(lon_sw):03d}_00_DEM"


def dem_tiles_for_bbox(bbox):
    lon_min, lat_min, lon_max, lat_max = bbox
    for la in range(int(np.floor(lat_min)), int(np.floor(lat_max - 1e-9)) + 1):
        for lo in range(int(np.floor(lon_min)), int(np.floor(lon_max - 1e-9)) + 1):
            yield la, lo


DEM_TILES = list(dem_tiles_for_bbox(BBOX))
print(f"Need up to {len(DEM_TILES)} DEM tiles "
      f"({len({t[0] for t in DEM_TILES})} lat × {len({t[1] for t in DEM_TILES})} lon)")

In [ ]:
dem_paths = []
missing = []
for la, lo in tqdm(DEM_TILES, desc="DEM tiles"):
    tile = dem_tile_id(la, lo)
    url  = f"{DEM_BASE}/{tile}/{tile}.tif"
    dst  = DEM_DIR / f"{tile}.tif"
    try:
        stream_download(url, dst)
        dem_paths.append(dst)
    except requests.HTTPError as e:
        if e.response is not None and e.response.status_code == 404:
            missing.append(tile)        # legitimate ocean tile
        else:
            raise

print(f"Got {len(dem_paths)} DEM tiles, {len(missing)} missing (ocean)")

In [ ]:
import rioxarray  # noqa: F401  (registers .rio accessor)
from osgeo import gdal

# Surface GDAL errors as Python exceptions instead of silent stderr prints.
gdal.UseExceptions()

# 1. Build a VRT mosaic. The VRT is a tiny XML file that GDAL treats as a
#    single logical raster; underlying COGs are streamed lazily.
#    We use the Python bindings (osgeo.gdal) instead of the gdalbuildvrt
#    executable — the .exe is not on PATH on most Windows installs even
#    when rasterio is working, and the bindings call the same library code.
VRT_PATH = DEM_DIR / "iberia_dem.vrt"
if not VRT_PATH.exists() or VRT_PATH.stat().st_size == 0:
    ds = gdal.BuildVRT(str(VRT_PATH), [str(p) for p in dem_paths])
    ds.FlushCache()
    ds = None                                # release the file handle
print(f"VRT mosaic: {VRT_PATH}  ({VRT_PATH.stat().st_size / 1e3:.1f} KB)")

# 2. Slope at native resolution from the same VRT. computeEdges=True gives
#    clean edges instead of a NoData border. scale=111120 is the m/° factor
#    (latitude direction); the longitude direction inherits a 1/cos(lat)
#    bias of ~18 % at lat 40°. Acceptable for this static covariate.
SLOPE_NATIVE = DEM_DIR / "slope_native.tif"
if not SLOPE_NATIVE.exists() or SLOPE_NATIVE.stat().st_size == 0:
    ds = gdal.DEMProcessing(
        str(SLOPE_NATIVE), str(VRT_PATH),
        processing="slope",
        options=gdal.DEMProcessingOptions(
            computeEdges=True,
            scale=111120,
        ),
    )
    ds.FlushCache()
    ds = None
print(f"Slope raster: {SLOPE_NATIVE}  "
      f"({SLOPE_NATIVE.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# 3. Reproject onto the project grid via rioxarray.
#    Build a target template DataArray. Internally we work in north-up
#    convention (y descending) to match rasterio/rioxarray; we flip back to
#    ascending lat at the end to match the project layout.
target = (
    xr.DataArray(
        np.zeros(DST_SHAPE, dtype=np.float32),
        dims=("y", "x"),
        coords={"y": LAT[::-1], "x": LON},
    )
    .rio.write_crs(DST_CRS)
    .rio.write_transform(DST_TRANSFORM)
)

# Open the VRT and the slope raster lazily, with Dask chunking so the full
# native-resolution mosaic never materialises.
elev_da = (
    rioxarray.open_rasterio(VRT_PATH, masked=True, chunks={"x": 3600, "y": 3600})
    .squeeze("band", drop=True)
)
slope_da = (
    rioxarray.open_rasterio(SLOPE_NATIVE, masked=True, chunks={"x": 3600, "y": 3600})
    .squeeze("band", drop=True)
)

# Average resampling on a single source: every destination cell with any
# valid land coverage gets a value. No per-tile boundary handling required.
mean_elev    = elev_da.rio.reproject_match(target, resampling=Resampling.average)
mean_elev_sq = (elev_da ** 2).rio.reproject_match(target, resampling=Resampling.average)
mean_slope   = slope_da.rio.reproject_match(target, resampling=Resampling.average)

# Variance identity. Clip at 0 to absorb the small numerical residual that
# can push var slightly negative for nearly-flat cells.
std_elev = np.sqrt((mean_elev_sq - mean_elev ** 2).clip(min=0))

# Trigger compute and report.
mean_elev_v = mean_elev.values
std_elev_v  = std_elev.values
slope_v     = mean_slope.values
denom_zeros = int(np.isnan(mean_elev_v).sum())

print(f"Elevation mean:  min {np.nanmin(mean_elev_v):7.1f} m, "
      f"max {np.nanmax(mean_elev_v):7.1f} m, mean {np.nanmean(mean_elev_v):7.1f} m")
print(f"Elevation std :  min {np.nanmin(std_elev_v):7.1f} m, "
      f"max {np.nanmax(std_elev_v):7.1f} m, mean {np.nanmean(std_elev_v):7.1f} m")
print(f"Slope mean    :  min {np.nanmin(slope_v):7.2f}°, "
      f"max {np.nanmax(slope_v):7.2f}°, mean {np.nanmean(slope_v):7.2f}°")
print(f"Cells with no land coverage (expected: ocean + bbox margin): "
      f"{denom_zeros} / {mean_elev_v.size}")

In [ ]:
# Re-orient onto the project's ascending-lat layout and assemble Dataset.
def to_grid(da, name, units, long_name):
    out = (
        da.rename({"y": "lat", "x": "lon"})
          .sortby("lat")
          .astype(np.float32)
          .rename(name)
    )
    # rioxarray leaves a `spatial_ref` scalar coord behind; drop it so the
    # NetCDF stays compatible with the climate cube.
    out = out.drop_vars("spatial_ref", errors="ignore")
    out.attrs.update(units=units, long_name=long_name)
    return out

dem_ds = xr.Dataset({
    "elev_mean":  to_grid(mean_elev,  "elev_mean",
                           "m", "Mean elevation"),
    "elev_std":   to_grid(std_elev,   "elev_std",
                           "m", "Sub-grid elevation standard deviation"),
    "slope_mean": to_grid(mean_slope, "slope_mean",
                           "degrees", "Mean slope (computed at 30 m via gdaldem)"),
})
dem_ds.attrs.update(
    title="Copernicus DEM GLO-30 — topographic statistics on Iberian 0.25° grid",
    source="https://copernicus-dem-30m.s3.amazonaws.com",
    version="GLO-30",
    processing="VRT mosaic of {n} tiles → rio.reproject_match (Resampling.average); "
               "slope from gdaldem at native resolution".format(n=len(dem_paths)),
    bbox=str(BBOX),
)

out_dem = PROC_DIR / "static_topography.nc"
dem_ds.to_netcdf(out_dem)
print(f"Wrote {out_dem}  ({out_dem.stat().st_size / 1e6:.1f} MB)")

# Visual check with explicit colour ranges so the colourbar is honest.
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
dem_ds.elev_mean.plot(ax=axes[0],  cmap="terrain", vmin=0, vmax=3500)
dem_ds.elev_std.plot(ax=axes[1],   cmap="magma",   vmin=0, vmax=600)
dem_ds.slope_mean.plot(ax=axes[2], cmap="cividis", vmin=0, vmax=30)
for ax, t in zip(axes, ["mean elevation [m]", "sub-grid std [m]",
                         "mean slope [°]"]):
    ax.set_title(t); ax.set_xlabel(""); ax.set_ylabel("")
plt.show()

## 3. MODIS NDVI — climatology and dynamic anomaly

MOD13A3 (Terra, monthly, 1 km, sinusoidal projection). NASA Earthdata Login
required — set up a free account at <https://urs.earthdata.nasa.gov/> and
either store credentials in `~/.netrc` or set the env vars
`EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD`. The `earthaccess` package
handles authentication, search, and download.

Iberia is covered by 4 sinusoidal tiles: `h17v04`, `h17v05`, `h18v04`,
`h18v05`. We download the full 2000-11 to 2025-12 record.

**Per-month processing**:
1. Read NDVI and Pixel Reliability subdatasets from each granule.
2. Mask to reliability ∈ {0 (good), 1 (marginal)}; drop fill (`-3000`).
3. Apply NDVI scale factor (× 0.0001) → physical [-0.2, 1.0].
4. Reproject from sinusoidal to the 0.25° lat/lon grid with the
   validity-mask trick (data with invalid pixels zeroed out, plus a separate
   mask channel for area-weighted averaging).

**Two outputs**:
- A static *climatology* — long-term mean by calendar month, computed from the
  training years only, kept on disk as a (12, lat, lon) array.
- A dynamic *anomaly* time series — monthly NDVI minus its month-of-year
  climatology, on the full record. Use this as the K=4 antecedent
  conditioning channel alongside the climate variables.

In [ ]:
import earthaccess

# Pick up credentials from ~/.netrc, or set EARTHDATA_USERNAME / EARTHDATA_PASSWORD.
auth = earthaccess.login(strategy="netrc")
assert auth.authenticated, "NASA Earthdata Login failed — check your credentials"

In [ ]:
# Search the full MOD13A3 record over the bbox.
results = earthaccess.search_data(
    short_name="MOD13A3",
    version="061",
    bounding_box=BBOX,
    temporal=("2000-11-01", "2025-12-31"),
)
print(f"Found {len(results)} MOD13A3 granules "
      f"(~{len(results) // 4} months × 4 tiles)")

In [ ]:
NDVI_DIR = RAW_DIR / "modis_ndvi"
NDVI_DIR.mkdir(parents=True, exist_ok=True)

# Download — earthaccess skips files already present.
ndvi_files = earthaccess.download(results, str(NDVI_DIR))
ndvi_files = sorted(Path(p) for p in ndvi_files)
print(f"Have {len(ndvi_files)} HDF files in {NDVI_DIR}")

In [ ]:
# MOD13A3 conventions.
NDVI_SCALE = 0.0001
NDVI_FILL  = -3000

DATE_RE = re.compile(r"\.A(\d{4})(\d{3})\.")

def parse_modis_date(name: str) -> datetime | None:
    m = DATE_RE.search(name)
    if not m:
        return None
    return datetime.strptime(f"{m.group(1)} {m.group(2)}", "%Y %j")


def find_subdataset(hdf_path, keyword):
    """Return the GDAL subdataset path whose description contains `keyword`."""
    ds = gdal.Open(str(hdf_path))
    if ds is None:
        raise IOError(f"gdal.Open failed: {hdf_path}")
    try:
        for sds_path, desc in ds.GetSubDatasets():
            if keyword.lower() in desc.lower():
                return sds_path
    finally:
        ds = None
    raise KeyError(f"No subdataset matching {keyword!r} in {hdf_path.name}")

def read_subdataset(sds_path):
    """Open via osgeo. Returns (array, Affine transform, WKT crs)."""
    ds = gdal.Open(sds_path)
    if ds is None:
        raise IOError(f"gdal.Open failed: {sds_path}")
    data = ds.ReadAsArray()
    gt   = ds.GetGeoTransform()
    proj = ds.GetProjection()
    ds = None
    return data, Affine.from_gdal(*gt), proj  # WKT string works as src_crs




# Group files by year-month — Iberia spans 4 sinusoidal tiles per month.
by_month: dict[tuple[int, int], list[Path]] = {}
for p in ndvi_files:
    d = parse_modis_date(p.name)
    if d is None:
        continue
    by_month.setdefault((d.year, d.month), []).append(p)

months = sorted(by_month)
print(f"Coverage: {months[0]} → {months[-1]} ({len(months)} months)")

In [ ]:
def reproject_month(tile_paths):
    sum_vals = np.zeros(DST_SHAPE, dtype=np.float64)
    sum_mask = np.zeros(DST_SHAPE, dtype=np.float64)

    for tile in tile_paths:
        ndvi_raw, tr, crs = read_subdataset(find_subdataset(tile, "1 km monthly NDVI"))
        qa, _, _          = read_subdataset(find_subdataset(tile, "1 km monthly pixel reliability"))

        valid = ((qa == 0) | (qa == 1)) & (ndvi_raw != NDVI_FILL)
        ndvi  = np.where(valid, ndvi_raw.astype(np.float32) * NDVI_SCALE, 0.0)

        sum_vals += reproject_to_target(ndvi,                  tr, crs)
        sum_mask += reproject_to_target(valid.astype(np.float32), tr, crs)

    with np.errstate(invalid="ignore", divide="ignore"):
        out = sum_vals / sum_mask
    out[sum_mask == 0] = np.nan
    return out.astype(np.float32)


# Build the full (time, lat, lon) NDVI cube. ~25 years × 12 months ≈ 300 frames.
times = [datetime(y, m, 15) for (y, m) in months]
ndvi_cube = np.full((len(months), *DST_SHAPE), np.nan, dtype=np.float32)

for i, (ym, paths) in enumerate(tqdm(list(by_month.items()),
                                      desc="NDVI months")):
    ndvi_cube[i] = reproject_month(paths)

In [ ]:
ndvi_da = xr.DataArray(
    ndvi_cube,
    dims=("time", "lat", "lon"),
    coords={"time": times, "lat": LAT, "lon": LON},
    name="ndvi",
    attrs={"long_name": "MOD13A3 monthly NDVI", "valid_range": [-0.2, 1.0]},
)

# ---- Train-window climatology --------------------------------------------
# Decision_log "Open questions": exact split TBD, but case-study years
# 2017 and 2022 belong in the test set. A defensible default training window
# avoiding both is 2001–2014; revise once the split is locked.
TRAIN_START, TRAIN_END = 2001, 2014

train = ndvi_da.sel(time=slice(f"{TRAIN_START}-01-01", f"{TRAIN_END}-12-31"))
clim = train.groupby("time.month").mean("time", skipna=True)
clim.name = "ndvi_climatology"
clim.attrs.update(
    long_name="MOD13A3 NDVI calendar-month climatology",
    train_window=f"{TRAIN_START}-{TRAIN_END}",
)

# Anomaly: NDVI minus its month-of-year climatology, full record.
anom = ndvi_da.groupby("time.month") - clim
anom.name = "ndvi_anom"
anom.attrs.update(
    long_name="NDVI anomaly w.r.t. train-window calendar-month climatology"
)

print(f"Climatology shape: {clim.shape}    "
      f"mean = {float(clim.mean()):.3f}")
print(f"Anomaly time range: {anom.time.values[0]} → {anom.time.values[-1]}")
print(f"Anomaly stats: mean = {float(anom.mean()):.3f}, "
      f"std = {float(anom.std()):.3f}")

In [ ]:
# Save: separate files for the static climatology and the dynamic anomaly.
out_clim = PROC_DIR / "static_ndvi_climatology.nc"
clim.to_dataset().to_netcdf(out_clim)
print(f"Wrote {out_clim}  ({out_clim.stat().st_size / 1e6:.1f} MB)")

out_dyn = PROC_DIR / "dynamic_ndvi_anomaly.nc"
xr.Dataset({"ndvi": ndvi_da, "ndvi_anom": anom}).to_netcdf(out_dyn)
print(f"Wrote {out_dyn}  ({out_dyn.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Sanity check: climatology by month + a couple of anomaly snapshots.
fig, axes = plt.subplots(3, 4, figsize=(14, 8), constrained_layout=True)
for m, ax in enumerate(axes.flat, start=1):
    clim.sel(month=m).plot(ax=ax, vmin=0, vmax=0.8, cmap="YlGn",
                            add_colorbar=False)
    ax.set_title(f"climatology — month {m}")
    ax.set_xlabel(""); ax.set_ylabel("")
fig.suptitle("MOD13A3 monthly NDVI climatology (train window)")
plt.show()

# Anomaly during the 2022 Iberian summer.
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, mo in zip(axes, [6, 7, 8]):
    anom.sel(time=f"2022-{mo:02d}-15").plot(
        ax=ax, vmin=-0.3, vmax=0.3, cmap="RdBu",
    )
    ax.set_title(f"NDVI anomaly 2022-{mo:02d}")
    ax.set_xlabel(""); ax.set_ylabel("")
plt.show()

## 4. Summary of outputs

```
data/processed/
├── static_landcover_fuelfractions.nc   # (fuel_class=7, lat=32, lon=54)
├── static_topography.nc                # elev_mean, elev_std, slope_mean
├── static_ndvi_climatology.nc          # (month=12, lat=32, lon=54)
└── dynamic_ndvi_anomaly.nc             # ndvi, ndvi_anom on (time, lat, lon)
```

The three static files are stack-ready alongside the ERA5 climate cube — same
grid, same coordinate convention, same NetCDF layout. The dynamic NDVI
anomaly file gets sliced into K = 4 antecedent monthly anomaly channels at
training time, alongside the climate conditioning stack.
